# 🧪 GIADA Task 32 v2 — feedback dinamico calcio–SK–voltaggio
La v1 resta archiviata come floor tecnicamente non ammissibile: CaDynamics faceva ricalcolare automaticamente E_Ca contro il contratto a reversal fisso. Questa v2 imposta e verifica E_Ca=120 mV a ogni passo, congela in anticipo il timing della sorgente e giudica i checkpoint congelati solo se il floor formula–NEURON passa sulle condizioni di conferma. Nessun training, sinapsi, morfologia o claim di speedup.


In [ ]:
from pathlib import Path
import base64,json,os,shutil,subprocess,sys,uuid
from IPython.display import display,Javascript
WORK=Path('/kaggle/working')/('giada_task32_'+uuid.uuid4().hex[:8]);WORK.mkdir()
REPO=WORK/'giada';TEACHER=WORK/'neuron_as_deep_net'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin',os.environ.get('GIADA_CODE_REVISION','codex/surrogate-validity-audit')],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
try:
 import neuron
except ImportError:
 subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
 import neuron
assert neuron.__version__.split('+')[0]=='8.2.7',neuron.__version__
import torch
assert torch.cuda.is_available(),'Task32 richiede CUDA per i checkpoint congelati'
print({'revision':REVISION,'teacher':'074c466','neuron':neuron.__version__,'gpu':torch.cuda.get_device_name()})


In [ ]:
OUTPUT=Path('/kaggle/working/artifacts')/('giada_task32_'+REVISION[:7]+'_'+uuid.uuid4().hex[:8])
completed=subprocess.run([sys.executable,'-u',str(REPO/'scripts/run_task32_dynamic_calcium_feedback.py'),'--output',str(OUTPUT),'--teacher',str(TEACHER)],check=False)
path=OUTPUT/('failure_report.json' if (OUTPUT/'failure_report.json').exists() else 'final_report.json')
report=json.loads(path.read_text()) if path.is_file() else {'valid':False,'error':f'worker interrupted: {completed.returncode}'}
display({k:report.get(k) for k in ('valid','native_floor_admissible','native_eca_max_deviation_mv','confirmation_formula_floor','scientific_primary_passed','source_hypothesis_frozen_before_run','native_calcium_range_mM','native_voltage_range_mv','error')})
if report.get('models'):
 display({family:{seed:{'primary':row['primary'],'diagnostic_80ms':row['diagnostic_80ms']} for seed,row in runs.items()} for family,runs in report['models'].items()})
print({'output':str(OUTPUT),'exit':completed.returncode})


In [ ]:
archive=Path(shutil.make_archive(str(Path('/kaggle/working')/OUTPUT.name),'zip',OUTPUT))
if os.environ.get('GIADA_MANUAL_DOWNLOAD','0')=='1':
 payload=base64.b64encode(archive.read_bytes()).decode('ascii')
 display(Javascript(f"const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"))
print({'archive':str(archive),'size_mib':round(archive.stat().st_size/2**20,2)})
